In [ ]:
# import os
# import sys
# project_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
# sys.path.append(project_path)
#
# requirements_path = os.path.join(project_path, "SECONDARY/requirements.txt")
# !{sys.executable} -m pip install -r "{requirements_path}"

In [ ]:
import os
import sys
import time
!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import datetime
from datetime import timedelta
from dateutil import parser
from SRC.LIBRARIES.new_data_utils import fetch
from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP
import SRC.LIBRARIES.new_fibonacci_plot_utils as nfpu
import SRC.LIBRARIES.new_fibonacci_statistics_utils as nfsu
import SRC.LIBRARIES.new_utils as nu

sym='BTC'
symbol = f'{sym}USDT'
discretization = '1H'
display_start_date_str = '2026-01-01 00:00' # дата, с которой вы хотите отображать '2026-03-25 14:30:00' -- формат даты и времени
load_end_date = datetime.now()# parser.parse("2025-12-31")
# load_end_date = parser.parse("2025-12-31")
measure_percentile = 0.40 # 0.5 для старого поведения
use_candle_size_instead_of_shadow=True
filter_by_measure_range=False           # включить/выключить фильтрацию по диапазону размера
measure_lower_mult = 0.5                # минимальный множитель относительно среднего
measure_upper_mult = 2.0                # максимальный множитель относительно среднего
use_mrc=True
use_mrc_r2=True
use_mrc_s2=True
use_stop_loss=True
capital_per_trade = 1000  # выделенный капитал на сделку
commission_rate = 0.00075
base_level = -0.618
# position_weights = [1]
# position_weights = [0.5, 0.5]
position_weights = [0.2, 0.3, 0.5]
# position_weights = [0.1, 0.2, 0.3, 0.4]
# position_weights = [0.15, 0.15, 0.25, 0.45]
# position_weights = [0.07, 0.12, 0.20, 0.28, 0.33]

if abs(sum(position_weights) - 1.0) > 1e-9:
    os.system('say "ERROR! ERROR! ERROR! Check position weights"')  # голосовое оповещение
    raise ValueError(f"position_weights sum {sum(position_weights)} is not equal 1")

levels = [base_level - (i-1)*1.0 for i in range(1, len(position_weights)+1)]   # [-0.618, -1.618, -2.618, ...]
sl_level = levels[-1] - 1.0   # стоп-лосс на 1.0 ниже последнего уровня
position_sizes = {}

for i, level in enumerate(levels):
    position_sizes[level] = capital_per_trade * position_weights[i]

market_type = 'SPOT'
mrc_buffer = 1000
rsi_buffer = 200
stoch_buffer = 17
macd_buffer = 200
atr_buffer = 200
display_start_dt = parser.parse(display_start_date_str)
start_time = time.perf_counter()

# Определяем таймфрейм в секундах
discretization_value = int(discretization[:-1])
timeframe_seconds = {
    'M': discretization_value * 60,
    'H': discretization_value * 3600,
    'D': discretization_value * 86400
}.get(discretization[-1], 1800)

# Рассчитываем дату начала загрузки с буфером
load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

mrc_df = fetch(market_type, symbol, discretization, load_start_dt, load_end_date)
df = mrc_df.iloc[mrc_buffer:].copy()
df_1m = df if discretization == "1M" else fetch(market_type, symbol, "1M", display_start_dt, load_end_date)

In [ ]:
import numpy as np
import pandas as pd

def find_target_candles(df_display, df_buffer, window=10):
    """
    Поиск целевых свечей по критериям:
    1. Касание линии R2 или S2 тенью/телом
    2. Объем выше перцентиля предыдущих свечей
    3. Размер свечи/тень выше перцентиля предыдущих свечей
    4. (опционально) Размер свечи/тень в диапазоне [lower_mult * avg, upper_mult * avg]
    """
    if df_display.index[0] in df_buffer.index:
        df_full = df_buffer.copy()
    else:
        df_full = pd.concat([df_buffer, df_display]).drop_duplicates().sort_index()

    # Скользящие перцентили для объёма (по предыдущим свечам)
    df_full['volume_percentile'] = df_full['volume'].shift(1).rolling(window=window, min_periods=1).quantile(measure_percentile)

    # В зависимости от флага рассчитываем меру и её перцентиль
    if use_candle_size_instead_of_shadow:
        df_full['measure'] = df_full['high'] - df_full['low']
    else:
        df_full['upper_shadow'] = df_full['high'] - df_full[['open', 'close']].max(axis=1)
        df_full['lower_shadow'] = df_full[['open', 'close']].min(axis=1) - df_full['low']
        df_full['measure'] = df_full[['upper_shadow', 'lower_shadow']].max(axis=1)

    df_full['measure_percentile'] = df_full['measure'].shift(1).rolling(window=window, min_periods=1).quantile(measure_percentile)

    # Для фильтрации по диапазону – скользящее среднее меры
    if filter_by_measure_range:
        df_full['measure_avg'] = df_full['measure'].shift(1).rolling(window=window, min_periods=1).mean()

    # Поиск целевых свечей
    target_conditions = []
    for idx in df_display.index:
        i = df_full.index.get_loc(idx)
        if isinstance(i, (slice, np.ndarray)):
            i = i.start if isinstance(i, slice) else i[0]

        if i < window:
            target_conditions.append(False)
            continue

        high = df_full['high'].iloc[i]
        low = df_full['low'].iloc[i]
        r2 = df_full['upband2'].iloc[i]
        s2 = df_full['loband2'].iloc[i]

        touches_r2 = (high >= r2 and low <= r2) or (low > r2)
        touches_s2 = (high >= s2 and low <= s2) or (high < s2)
        touches_level = not use_mrc or (use_mrc_r2 and touches_r2) or (use_mrc_s2 and touches_s2)

        volume_above = df_full['volume'].iloc[i] > df_full['volume_percentile'].iloc[i]
        measure_above = df_full['measure'].iloc[i] > df_full['measure_percentile'].iloc[i]

        # Дополнительная фильтрация по диапазону (если включена)
        range_ok = True
        if filter_by_measure_range:
            avg = df_full['measure_avg'].iloc[i]
            if not pd.isna(avg):
                low_bound = avg * measure_lower_mult
                high_bound = avg * measure_upper_mult
                current_measure = df_full['measure'].iloc[i]
                range_ok = (low_bound <= current_measure <= high_bound)

        target_conditions.append(touches_level and volume_above and measure_above and range_ok)

    df_display['is_target_candle'] = target_conditions
    return df_display

# Убеждаемся, что индексы уникальны
mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
df = df[~df.index.duplicated(keep='first')]

# 1. RSI
rsi_calc_df = nu.prepare_buffer_data(mrc_df, df, rsi_buffer)
df = nu.rsi(df, rsi_calc_df)

# 2. Stochastic
stoch_calc_df = nu.prepare_buffer_data(mrc_df, df, stoch_buffer)
df = nu.stochastic_tradingview(df, stoch_calc_df)

# 3. MACD
macd_calc_df = nu.prepare_buffer_data(mrc_df, df, macd_buffer)
df, macd = nu.macd(df, macd_calc_df)

# 4. ATR
atr_calc_df = nu.prepare_buffer_data(mrc_df, df, atr_buffer)
df = nu.atr(df, atr_calc_df)

In [ ]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import SRC.LIBRARIES.new_indicator_plot_utils as nipu

is_log_scale_by_default=True
candlestick_row = 1
volume_row = 2
rsi_row = 3
stoch_row = 4
macd_row = 5
atr_row = 6

fig_main = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    row_heights=[3, 0.7, 1, 1, 1.5, 1],
    vertical_spacing=0.03
)

fig_main.add_trace(
    go.Candlestick(
        x=df[_KIEV_TIMESTAMP],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

mrc_df = nu.mrc_calculate(mrc_df, df)
nipu.add_mrc(candlestick_row, fig_main, df)
nipu.add_volume(df, volume_row, fig_main)
nipu.add_rsi(rsi_row, fig_main, df)
nipu.add_stoch(stoch_row, fig_main, df)
nipu.add_macd(macd, df, macd_row, fig_main)
nipu.add_atr(atr_row, fig_main, df)

df = find_target_candles(df, mrc_df)
tcs_count = nfpu.add_target_candle_scatter('high', 1.005, "🐵", 'white', "down", df, fig_main, candlestick_row)

price_log_scale_value="log"
price_linear_scale_value="linear"
price_log_scale_title="Price (log scale)"
price_linear_scale_title="Price (linear scale)"

fig_main.update_layout(
    title=f"🐵{discretization}🐵 TCs: {tcs_count}",
    xaxis_rangeslider_visible=False,
    yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
    yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
    yaxis2_title="Volume",
    yaxis3_title="RSI",
    yaxis4_title="Stoch",
    yaxis5_title="MACD",
    yaxis6_title="ATR",
    height=1200,
    bargap=0,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=1 if is_log_scale_by_default else 0,
            x=0.115,
            y=1.073,
            buttons=[
                dict(
                    label="Linear",
                    method="relayout",
                    args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]
                ),
                dict(
                    label="Log",
                    method="relayout",
                    args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}]
                )
            ],
            font=dict(
                color="red",
                size=12,
                family="Arial"
            ),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="red",
            borderwidth=1
        )
    ]
)

stats = nfsu.analyze_and_print_statistics(df, df_1m, start_time, levels, sl_level, commission_rate, position_sizes, capital_per_trade, use_stop_loss, symbol, discretization, display_start_date_str, load_end_date, measure_percentile, use_candle_size_instead_of_shadow, filter_by_measure_range, measure_lower_mult, measure_upper_mult, use_mrc, use_mrc_r2, use_mrc_s2)
nfpu.plot_all_target_candles_multiframe_and_save_htmls_to_disc(df_trading=df, df_1m=df_1m, trading_timeframe=discretization, levels=levels, sl_level=sl_level, commission_rate=commission_rate, position_sizes=position_sizes, capital_per_trade=capital_per_trade, use_stop_loss=use_stop_loss, future_bars=500, output_dir="target_candles", clean_dir=True)
fig_main.show()

# os.system('say "Done"')  # голосовое оповещение
# или
os.system('afplay /System/Library/Sounds/Glass.aiff')  # воспроизведение системного звука
print('\a')